In [7]:
# --- 1. DATA CLEANSING (DUPLICATES & MISSING VALUE COUNT) ---

print("=" * 70)
print("1. DATA CLEANSING: DUPLICATES AND MISSING VALUE REPORT")
print("=" * 70)

# 1a. Remove Duplicates
initial_train_rows = X_train.shape[0]
X_train.drop_duplicates(inplace=True)
y_train = y_train[X_train.index] # Align y_train index after dropping rows
rows_dropped = initial_train_rows - X_train.shape[0]

print(f"Original X_train shape: ({initial_train_rows}, {X_train.shape[1]})")
print(f"Total duplicate rows removed from training set: {rows_dropped}")
print(f"New X_train shape after duplicate removal: {X_train.shape}")
print("-" * 35)

# 1b. Detailed Missing Value Report
print("Missing Value Report (before Imputation):")
missing_train = X_train.isnull().sum()
missing_test = X_test.isnull().sum()

missing_df = pd.DataFrame({
    'Train_Missing_Count': missing_train,
    'Train_Missing_%': (missing_train / X_train.shape[0]) * 100,
    'Test_Missing_Count': missing_test,
    'Test_Missing_%': (missing_test / X_test.shape[0]) * 100
})

missing_df = missing_df[missing_df['Train_Missing_Count'] > 0].sort_values(by='Train_Missing_%', ascending=False)

if not missing_df.empty:
    print("\n[FEATURES WITH MISSING VALUES]")
    print(missing_df.to_string())
    print(f"\nTotal features with missing data: {len(missing_df)}")
else:
    print("No missing values found in the training dataset. Imputation is not strictly necessary.")
print("-" * 70)

1. DATA CLEANSING: DUPLICATES AND MISSING VALUE REPORT
Original X_train shape: (204277, 16)
Total duplicate rows removed from training set: 0
New X_train shape after duplicate removal: (204277, 16)
-----------------------------------
Missing Value Report (before Imputation):
No missing values found in the training dataset. Imputation is not strictly necessary.
----------------------------------------------------------------------


In [8]:
# --- 2. IMPUTATION (MISSING VALUES) ---
from sklearn.compose import ColumnTransformer

print("=" * 70)
print("2. IMPUTATION: FILLING MISSING VALUES")
print("=" * 70)

print(f"Imputation Strategy Details:")
print(f"  - Numerical Columns ({len(numerical_cols)}): Median imputation.")
print(f"  - Categorical Columns ({len(categorical_cols)}): Most Frequent (Mode) imputation.")

# Define Imputation Transformers
numerical_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

# Create Imputation ColumnTransformer
imputation_preprocessor = ColumnTransformer(
    transformers=[
        ('num_impute', numerical_imputer, numerical_cols),
        ('cat_impute', categorical_imputer, categorical_cols)
    ],
    remainder='passthrough'
)

# Apply Imputation and convert back to DataFrame
imputation_preprocessor.fit(X_train)
X_train_imputed_array = imputation_preprocessor.transform(X_train)
X_test_imputed_array = imputation_preprocessor.transform(X_test)

# Reconstruct DataFrame (for easy chaining)
imputed_feature_names = list(numerical_cols) + list(categorical_cols)
X_train_imputed = pd.DataFrame(X_train_imputed_array, columns=imputed_feature_names)
X_test_imputed = pd.DataFrame(X_test_imputed_array, columns=imputed_feature_names)

print("-" * 35)
print(f"✅ Imputation complete.")
print(f"X_train imputed shape: {X_train_imputed.shape}")
print(f"X_test imputed shape: {X_test_imputed.shape}")
print(f"Total NaNs in X_train_imputed (must be 0): {X_train_imputed.isnull().sum().sum()}")
print("-" * 70)

2. IMPUTATION: FILLING MISSING VALUES
Imputation Strategy Details:
  - Numerical Columns (9): Median imputation.
  - Categorical Columns (7): Most Frequent (Mode) imputation.
-----------------------------------
✅ Imputation complete.
X_train imputed shape: (204277, 16)
X_test imputed shape: (51070, 16)
Total NaNs in X_train_imputed (must be 0): 0
----------------------------------------------------------------------


In [9]:
# --- 3. FEATURE SCALING AND ENCODING ---

print("=" * 70)
print("3. SCALING AND ENCODING")
print("=" * 70)

print(f"Transformation Strategy Details:")
print(f"  - Numerical Features ({len(numerical_cols)}): StandardScaler applied.")
print(f"  - Categorical Features ({len(categorical_cols)}): OneHotEncoder applied.")

# Define Scaling and Encoding Transformers
num_scaler = StandardScaler()
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Create Scaling and Encoding ColumnTransformer
scaling_encoding_preprocessor = ColumnTransformer(
    transformers=[
        ('scaling', num_scaler, numerical_cols),
        ('encoding', cat_encoder, categorical_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=True
).set_output(transform="pandas")

# Apply Transformation on IMPUTED Data
X_train_final = scaling_encoding_preprocessor.fit_transform(X_train_imputed)
X_test_final = scaling_encoding_preprocessor.transform(X_test_imputed)

# Get all feature names after encoding
final_feature_names = X_train_final.columns.tolist()
final_feature_count = len(final_feature_names)

print("-" * 35)
print(f"✅ Scaling and Encoding complete. Total features created: {final_feature_count}")
print(f"X_train final shape: {X_train_final.shape}")
print(f"Example of final features (first 10):")
print(final_feature_names[:10])
print("-" * 35)

# Encode Target (Label Encoding)
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Target variable (y) Label Encoding applied:")
print(f"Original classes mapped to: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")
print("-" * 70)

3. SCALING AND ENCODING
Transformation Strategy Details:
  - Numerical Features (9): StandardScaler applied.
  - Categorical Features (7): OneHotEncoder applied.
-----------------------------------
✅ Scaling and Encoding complete. Total features created: 31
X_train final shape: (204277, 31)
Example of final features (first 10):
['scaling__Age', 'scaling__Income', 'scaling__LoanAmount', 'scaling__CreditScore', 'scaling__MonthsEmployed', 'scaling__NumCreditLines', 'scaling__InterestRate', 'scaling__LoanTerm', 'scaling__DTIRatio', "encoding__Education_Bachelor's"]
-----------------------------------
Target variable (y) Label Encoding applied:
Original classes mapped to: {0: 0, 1: 1}
----------------------------------------------------------------------


In [11]:
# --- 4. BINNING (DISCRETIZATION) ---

BINNING_COL = 'Age' # Use an appropriate numerical column

print("=" * 70)
print(f"4. BINNING (DISCRETIZATION) FOR '{BINNING_COL}'")
print("=" * 70)

if BINNING_COL in X_train_imputed.columns:
    
    # Create the binned feature on imputed data (TRAINING SET)
    # 1. Use retbins=True to capture the automatically calculated numerical bin boundaries (bins_calculated)
    _, bins_calculated = pd.cut(
        X_train_imputed[BINNING_COL], 
        bins=5, # Using 5 equal width bins
        labels=[f'Bin_{i}' for i in range(1, 6)], 
        include_lowest=True,
        retbins=True # <--- Captures the numerical boundaries
    )

    # 2. Re-apply the cut using the numerical boundaries to get the binned feature
    X_train_imputed[f'{BINNING_COL}_Binned'] = pd.cut(
        X_train_imputed[BINNING_COL], 
        bins=bins_calculated, # Use the numerical array
        labels=[f'Bin_{i}' for i in range(1, 6)], 
        include_lowest=True
    )
    
    # Apply the SAME numerical boundaries to the test set
    X_test_imputed[f'{BINNING_COL}_Binned'] = pd.cut(
        X_test_imputed[BINNING_COL], 
        bins=bins_calculated, # <--- Reuse the same numerical boundaries
        labels=[f'Bin_{i}' for i in range(1, 6)], 
        include_lowest=True
    )

    print(f"✅ '{BINNING_COL}' successfully binned into 5 equal-width categories.")
    print("\nNew Binned Feature Distribution (Training Set):")
    print(X_train_imputed[f'{BINNING_COL}_Binned'].value_counts(dropna=False).to_string())
    print("\nNOTE: This new feature must be included in the final encoding step (Block 3).")
else:
    print(f"Skipped Binning: '{BINNING_COL}' column not available after imputation.")
print("-" * 70)

4. BINNING (DISCRETIZATION) FOR 'Age'
✅ 'Age' successfully binned into 5 equal-width categories.

New Binned Feature Distribution (Training Set):
Age_Binned
Bin_5    43093
Bin_1    42956
Bin_4    39550
Bin_2    39526
Bin_3    39152

NOTE: This new feature must be included in the final encoding step (Block 3).
----------------------------------------------------------------------


In [ ]:
# --- 5. TARGET ENCODING WITH CAUTION ---

# Select a high-cardinality categorical column (e.g., the first one available)
HIGH_CARD_COL = categorical_cols[0] if len(categorical_cols) > 0 else None

print("=" * 70)
print(f"5. TARGET ENCODING FOR '{HIGH_CARD_COL}' (CAUTIONARY STEP)")
print("=" * 70)

if HIGH_CARD_COL and HIGH_CARD_COL in X_train_imputed.columns:
    
    # Calculate mean target on training data only
    target_map = pd.Series(y_train_encoded).groupby(X_train_imputed[HIGH_CARD_COL]).mean()
    overall_mean = pd.Series(y_train_encoded).mean()
    
    # Apply Target Encoding
    X_train_imputed[f'{HIGH_CARD_COL}_TargetEncoded'] = X_train_imputed[HIGH_CARD_COL].map(target_map)
    X_test_imputed[f'{HIGH_CARD_COL}_TargetEncoded'] = X_test_imputed[HIGH_CARD_COL].map(target_map).fillna(overall_mean)
    
    # Drop the original high-cardinality column to avoid multicollinearity/overfitting
    X_train_imputed.drop(columns=[HIGH_CARD_COL], inplace=True)
    X_test_imputed.drop(columns=[HIGH_CARD_COL], inplace=True)
    
    print(f"✅ Target Encoding applied to '{HIGH_CARD_COL}'.")
    print(f"Overall Target Mean (Fallback for new categories): {overall_mean:.4f}")
    
    print("\nTop 5 Categories by Target Mean:")
    print(target_map.sort_values(ascending=False).head().to_string())
    print(f"\nNOTE: Original feature '{HIGH_CARD_COL}' has been replaced/dropped.")
else:
    print(f"Skipped Target Encoding: No suitable categorical column found or already processed.")
print("-" * 70)

5. TARGET ENCODING FOR 'Education' (CAUTIONARY STEP)
✅ Target Encoding applied to 'Education'.
Overall Target Mean (Fallback for new categories): 0.1161

Top 5 Categories by Target Mean:
Education
High School    0.128516
Bachelor's     0.120632
Master's       0.108906
PhD            0.106317

NOTE: Original feature 'Education' has been replaced/dropped.
----------------------------------------------------------------------


In [16]:
import numpy as np
import pandas as pd # Assuming this is available
# No need for matplotlib or seaborn if you only want the text output

# --- 6. ADDRESSING MULTICOLLINEARITY (Correlation Check - Simple Version) ---

print("=" * 70)
print("6. SIMPLE MULTICOLLINEARITY CHECK (THRESHOLDING)")
print("=" * 70)

# Re-extract numerical columns (assuming X_train_imputed is your DataFrame)
current_numerical_cols = X_train_imputed.select_dtypes(include=['float64', 'int64']).columns
correlation_matrix = X_train_imputed[current_numerical_cols].corr()

# Identify highly correlated pairs (Absolute Correlation > 0.95 for simplicity)
THRESHOLD = 0.95
# Use np.triu to select the upper triangle of the matrix (excluding the diagonal)
upper = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
highly_correlated_pairs = []

# Iterate and append pairs that exceed the threshold
for column in upper.columns:
    # Use idxmax to find the row index (feature name) with the maximum absolute value
    max_corr_abs = upper[column].abs().max()
    if max_corr_abs > THRESHOLD:
        col2 = upper[column].abs().idxmax() # The other column name
        corr = upper.loc[col2, column]      # The actual correlation value
        highly_correlated_pairs.append((col2, column, corr))
        
# Note: The original iteration method also works fine, but this is a slightly cleaner way 
# to ensure we capture *all* pairs above the threshold. We'll stick to the original loop structure 
# as it's clear and less prone to edge cases if done correctly:

highly_correlated_pairs = []
for i in range(len(current_numerical_cols)):
    for j in range(i + 1, len(current_numerical_cols)):
        col1 = current_numerical_cols[i]
        col2 = current_numerical_cols[j]
        corr = correlation_matrix.loc[col1, col2]
        
        if abs(corr) > THRESHOLD:
            highly_correlated_pairs.append((col1, col2, corr))


print(f"\nHighly correlated pairs (Absolute Correlation > {THRESHOLD}):")
if highly_correlated_pairs:
    for col1, col2, corr in highly_correlated_pairs:
        print(f"  - {col1} and {col2}: Correlation = {corr:.4f} (Consider dropping one)")
else:
    print("  - No highly correlated numerical pairs found above the threshold.")
print("-" * 70)

6. SIMPLE MULTICOLLINEARITY CHECK (THRESHOLDING)

Highly correlated pairs (Absolute Correlation > 0.95):
  - No highly correlated numerical pairs found above the threshold.
----------------------------------------------------------------------


In [17]:
# --- 7. HANDLING CLASS IMBALANCE (SMOTE) ---
from imblearn.over_sampling import SMOTE

print("=" * 70)
print("7. HANDLING CLASS IMBALANCE WITH SMOTE")
print("=" * 70)

# Check original distribution
original_counts = pd.Series(y_train_encoded).value_counts()
print(f"Original Training Class Distribution:")
print(f"  Class 0 (Majority): {original_counts.iloc[0]} samples")
print(f"  Class 1 (Minority): {original_counts.iloc[1]} samples")
print(f"  Imbalance Ratio: {original_counts.iloc[0] / original_counts.iloc[1]:.2f}:1")

# Apply SMOTE to the final processed data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_final, y_train_encoded)

# Check new distribution
smote_counts = pd.Series(y_train_smote).value_counts()
print("\nSMOTE Resampling Result:")
print(f"  Class 0 (Majority): {smote_counts.iloc[0]} samples")
print(f"  Class 1 (Minority): {smote_counts.iloc[1]} samples")

print(f"\n✅ SMOTE Resampling Complete.")
print(f"Final X_train shape for modeling: {X_train_smote.shape}")
print(f"Final y_train shape for modeling: {y_train_smote.shape}")
print("-" * 70)

7. HANDLING CLASS IMBALANCE WITH SMOTE
Original Training Class Distribution:
  Class 0 (Majority): 180555 samples
  Class 1 (Minority): 23722 samples
  Imbalance Ratio: 7.61:1

SMOTE Resampling Result:
  Class 0 (Majority): 180555 samples
  Class 1 (Minority): 180555 samples

✅ SMOTE Resampling Complete.
Final X_train shape for modeling: (361110, 31)
Final y_train shape for modeling: (361110,)
----------------------------------------------------------------------
